# Lab 6 — Notebook 2: TPC-H Q1, the Lab 5 callback

**What you'll do here:** rewrite your Lab 5 Hadoop MapReduce Q1 job as a PySpark DataFrame chain (about ten lines), and as a SparkSQL query. You'll read the physical plan and identify the **partial + final HashAggregate** that maps exactly onto the Combiner + Reducer you hand-wrote in Java.

Recall the Q1 SQL specification (also in your Lab 5 handout):

```sql
SELECT l_returnflag, l_linestatus,
       SUM(l_quantity) AS sum_qty,
       SUM(l_extendedprice) AS sum_base_price,
       SUM(l_extendedprice * (1 - l_discount)) AS sum_disc_price,
       SUM(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge,
       AVG(l_quantity) AS avg_qty,
       AVG(l_extendedprice) AS avg_price,
       AVG(l_discount) AS avg_disc,
       COUNT(*) AS count_order
FROM lineitem
WHERE l_shipdate <= DATE '1998-12-01' - INTERVAL '90' DAY
GROUP BY l_returnflag, l_linestatus
ORDER BY l_returnflag, l_linestatus;
```


## Setup

Same SparkSession boilerplate as Notebook 1. (If your Notebook 1 kernel is still running, you can re-use it by skipping this cell — but a fresh kernel is recommended so plan-inspection results are clean.)


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Lab6-N2-Q1")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.requester.pays.enabled", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-1.amazonaws.com")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

lineitem = spark.read.parquet("s3a://tpch-torstengrabs-parquet/1GB/lineitem/")
print(f"Loaded lineitem — {len(lineitem.columns)} columns")


## 3.1 Q1 in the DataFrame API (10 pts)

Translate the Q1 SQL above into a DataFrame chain. Use:

- `.filter()` for the `WHERE l_shipdate <= '1998-09-02'` clause (`'1998-12-01' - 90 days` = `'1998-09-02'`)
- `.groupBy("l_returnflag", "l_linestatus")`
- `.agg(...)` with `F.sum`, `F.avg`, `F.count` (deck slide 28)
- `.orderBy(...)` for the final sort
- `.alias(...)` on each aggregate to match the column names in the spec

Aim for about **10 lines of code**. Recall: your Lab 5 Java implementation was around 200 lines across 5 files.


In [ ]:
# TODO: implement Q1 as a DataFrame chain.
# q1_df = (lineitem
#     .filter(...)
#     .groupBy(...)
#     .agg(
#         F.sum(...).alias(...),
#         ...
#     )
#     .orderBy(...))
# q1_df.show(truncate=False)


**Expected output:** 4 rows. The grouping keys `(l_returnflag, l_linestatus)` take 4 distinct combinations: `(A, F)`, `(N, F)`, `(N, O)`, `(R, F)`. Sums and counts in the high-100-millions range for `count_order`, billions for `sum_extendedprice`-derived columns. The wall-time should be 30–60 s.


## 3.2 Read the physical plan (10 pts)

Now ask Catalyst to show you what it's actually going to do.


In [ ]:
q1_df.explain(mode="formatted")


Find these nodes in the plan (reading bottom-up):

1. **`Scan parquet`** — the leaf node. Note the `PushedFilters` and `ReadSchema` lines: Catalyst pushed the date filter and the column projection into the parquet reader.
2. **`Filter`** — re-applies the predicate (Catalyst can't always trust the pushed-down filter to exactly match SQL semantics, so it re-checks).
3. **`HashAggregate(... functions=[partial_sum(...), partial_count(...), ...])`** — the **partial** aggregate. This runs on each executor over its local partition. **This is what your Lab 5 Combiner did.**
4. **`Exchange hashpartitioning(l_returnflag, l_linestatus, ...)`** — the **shuffle**. Partial results are repartitioned by the group-by keys.
5. **`HashAggregate(... functions=[sum(...), count(...), ...])`** — the **final** aggregate. **This is what your Lab 5 Reducer did.**
6. **`Sort`** — the `ORDER BY`.


## 3.3 Map the plan stages to your Lab 5 code (10 pts)

In Lab 5 you wrote `Q1Mapper.java`, `Q1Combiner.java`, `Q1Reducer.java`, and a custom `Q1Value` Writable. In the markdown cell below, fill in the table mapping each Lab 5 class to the Catalyst plan node above.


| Lab 5 class | Catalyst plan node | What it does |
|---|---|---|
| `Q1Mapper` | TODO | TODO |
| `Q1Combiner` | TODO | TODO |
| (Hadoop shuffle) | TODO | TODO |
| `Q1Reducer` | TODO | TODO |


## 3.4 Q1 in SparkSQL (10 pts)

Same query, expressed as literal SQL via `spark.sql(...)` (deck slide 30). You'll need to:

1. Register `lineitem` as a SQL view with `createOrReplaceTempView`.
2. Pass the SQL text from the top of this notebook to `spark.sql(...)`. The DataFrame returned can be `.show()` like any other.


In [ ]:
# TODO: register lineitem as a SQL view, then write the Q1 SQL.
# lineitem.createOrReplaceTempView(...)
# q1_sql = spark.sql("""
#     SELECT ...
#     FROM lineitem
#     WHERE ...
#     GROUP BY ...
#     ORDER BY ...
# """)
# q1_sql.show(truncate=False)


## 3.5 Compare the two plans (5 pts)

Print the SparkSQL physical plan and compare to 3.2. They should be **physically identical** (same nodes, same order). The DataFrame API and SparkSQL are two surfaces over the same Catalyst engine.


In [ ]:
q1_sql.explain(mode="formatted")


**Question for your reflection** (you'll answer this in Part 5): if the two surfaces produce identical plans, what factors would lead you to prefer one over the other in production code?


## 3.6 Narrow vs. wide dependencies on the Q1 plan (5 pts)

Look at the **`.explain(mode="formatted")` output you printed in 3.2** (re-run that cell if you've cleared it) and answer the three questions below directly from that plan text. **Do not** read these answers off the Spark UI's Stages tab — the UI can show some stages as "skipped" when Spark reuses shuffle output from a prior cell run, which is confusing. The plan text is deterministic: it shows the same operator tree regardless of execution history, so count `Exchange` nodes and read the operators around them.

**Questions — answer from the 3.2 plan output, in the markdown cell below:**

1. How many **stages** does the Q1 job have? How can you tell from the plan?
2. Where exactly is the **stage boundary** (i.e., the wide dependency / shuffle) in the plan? Which transformation in your Q1 DataFrame chain creates it?
3. Which operators in the plan are part of the **narrow** lineage (pipelinable within a single stage)?

**Separate deliverable (not graded — just submitted as proof your job ran):** open the Spark UI (http://localhost:4040), find the Job for Q1, and save its DAG visualization as `notebook2_screenshots/q1_dag.png`.


**Answers** (replace this paragraph with yours):

1. _TODO_
2. _TODO_
3. _TODO_


## Closing — stop the session


In [ ]:
spark.stop()
